In [ ]:
#| echo: false
import os

import retentioneering as rete
from IPython.display import HTML

# Rendered by Quarto, slides have no Python kernel behind them, so every widget
# is baked in as a self-contained static page (render_static) instead of a live
# anywidget, with its settings sidebar closed to leave the slide to the
# chart. Run in Jupyter, the widgets stay live.
if "QUARTO_DOCUMENT_PATH" in os.environ:
    from retentioneering.widgets._base import RetentioneeringWidget

    # A widget can ask for its own height on a slide: `widget.slide_height = …`
    # (the widget's layout follows its `height`, the iframe follows the page).
    SLIDE_WIDGET_HEIGHT = 470

    def _static_bundle(self, include=None, exclude=None, **kwargs):
        height = getattr(self, "slide_height", None)
        if height:
            self.height = height
        return {
            "text/html": self.render_static(
                height=height or SLIDE_WIDGET_HEIGHT, sidebar_open=False
            ).data
        }

    RetentioneeringWidget._repr_mimebundle_ = _static_bundle

## About us

### Vladimir Kukushkin

![](img/vladimir-kukushkin.jpg)

- 13 years in product analytics
- ex. Member of Techincal Staff @ Perplexity
- Principal AI Quality Analyst @ Finom
- Co-founder of Retentioneering

### Maxim Godzi

![](img/maxim-godzi.jpg)

- PhD in XXX
- NN years in product analytics
- ex. Principal Data Science Manager @ Microsoft
- Co-founder of Retentioneering

# Part 1: Motivating example

## How does a team adopt a task tracker?

- **Taskboard**, a task tracker for teams: 7,089 users in 5,000 workspaces over eight weeks, 48,130 sessions, 391,680 events of 29 types.
- **Free for two, paid for teams**: a workspace owner can buy a Team plan to extend their team.
- The business needs owners to bring a team in, and to grow it past two.

In [ ]:
#| echo: false
#| output: false
import pandas as pd

# A static snapshot of taskboard.load_taskboard(): 5,000 workspaces, seed 1.
taskboard = rete.Eventstream(
    pd.read_csv("taskboard.csv.gz", parse_dates=["timestamp"]),
    schema={
        "path_cols": ["user_id", "session_id"],
        "segment_cols": ["channel", "platform"],
        "custom_cols": ["workspace_id"],
    },
)


In [ ]:
#| echo: false
from toy import app_mockups

HTML(app_mockups())


## Events and the happy path

| Getting started | Tasks | Discussion | Team | Power features | Billing |
|---|---|---|---|---|---|
| `signup` | `open_board` | `comment` | `invite_member` | `view_report` | `view_pricing` |
| `tour_step` | `create_task` | `mention` | `accept_invite` | `export` | `select_plan` |
| `template_view` | `edit_task` | `view_notifications` | `teammate_joined` | `create_automation` | `start_checkout` |
| `create_project` | `complete_task` |  |  | `connect_integration` | `enter_payment` |
|  | `assign_task` |  |  |  | `payment_error` |
|  | `set_due_date` |  |  |  | `support_ticket` |
|  | `add_dependency` |  |  |  | `upgrade` |
|  | `view_timeline` |  |  |  |  |

**Owner's happy path:** `signup` → `create_project` → `invite_member` → `teammate_joined` → `invite_member` → `view_pricing` → `upgrade`


## (Almost) every path is unique

In [ ]:
#| echo: false
from toy import (
    branching_figure,
    distinct_colors,
    first_visits_figure,
    stage_graph,
    taskboard_colors,
    visit_paths_figure,
)

USERS = ["user_00089", "user_00021", "user_00059", "user_00015"]
# The trees: a random 1,000 of the 7,089 users is enough to see the point.
sample = taskboard.sample_paths(n=1000, random_state=7)
event_colors_31 = distinct_colors(taskboard.df["event"])
HTML(
    first_visits_figure(taskboard, USERS, colors=event_colors_31)
    + branching_figure(sample, 15, colors=event_colors_31, size=250)
)

## The transition graph

In [ ]:
#| echo: false
from toy import prefix_tree_svg, short_paths_events, short_paths_html

short = rete.Eventstream(short_paths_events())
HTML(short_paths_html())


In [ ]:
#| echo: false
HTML(prefix_tree_svg(short, 4, size=(360, 300)))


In [ ]:
#| echo: false
short_graph = short.transition_graph(
    edge_weight="proba_out", label_scale=2.5, view={"edgeFilter": {"mode": "range", "range": [0, 1]}}
)
short_graph.slide_height = 320
short_graph


- One of the core tools of Retentioneering; another view of the customer journey
- A tree branches on every distinct beginning; the graph merges the paths
- One node per event, one edge per pair of neighbors
- By default, edge weights are Markov chain probabilities – P(Y | X); other weights are supported
- E.g. outgoing edges for `A`: P(B | A) = P(C | A) = 2/5, a loop: P(A | A) = 1/5

## Even the graph is tangled on real data

In [ ]:
#| echo: false
stage_graph(taskboard, view={"edgeFilter": {"mode": "range", "range": [0, 1]}}, slide_height=330)

In [ ]:
#| echo: false
from taskboard import EVENT_GROUP

# Filtering: the events under 1% of the log, dropped.
shares = taskboard.df.loc[taskboard.df["event_type"] == "raw", "event"].value_counts(normalize=True)
rare = list(shares[shares < 0.01].index)
stage_graph(taskboard.drop_events(rare), view={"edgeFilter": {"mode": "range", "range": [0, 1]}}, slide_height=330)


In [ ]:
#| echo: false
# Truncating: the first ten steps of each path.
first_ten = taskboard.truncate_paths(
    start_anchor="path_start", end_anchor={"pattern": "path_start", "offset": 10}
)
stage_graph(first_ten, view={"edgeFilter": {"mode": "range", "range": [0, 1]}}, slide_height=330)


In [ ]:
#| echo: false
# Grouping: one event per part of the app.
stage_graph(taskboard.rename_events(EVENT_GROUP), view={"edgeFilter": {"mode": "range", "range": [0, 1]}}, slide_height=330)


The graph looks messy even applied to the Taskboard dataset (29 unique events)

Common ways to simplify sequence log data:

- **Filter out rare events**: drop the 12 events under 1% of the log, `invite_member`, `teammate_joined` and `upgrade` among them
- **Truncate paths**: keep only the first ten steps of each path
- **Group events**: one node per part of the app, `tasks` for all eight task events

All three distort the meaning of the clickstream: transitions that never happened, actions that can't be told apart, journeys cut short

## Idea: one session, one event

In [ ]:
#| echo: false
#| output: false
import json

# The names the next slides give the clusters (checked against the profiles).
SESSIONS = ["teamwork", "focus", "onboarding", "checkout", "checkout_failed", "review", "short", "planning"]
SESSION_NAMES = {f"cluster_{i}": name for i, name in enumerate(SESSIONS)}

# Sessions by their bag of events: the counts of all the events in them.
sessions = (
    taskboard
        .add_clusters(
            "session_type", features=[{"metric": "event_count_bulk"}],
            method_args={"n_clusters": 8, "n_init": 30}, scaler="std", path_col="session_id",
        )
        .collapse_events(group_col="session_id", name={"col": "session_type"})
        .rename_events(SESSION_NAMES)
)
# Users by the kinds of sessions they make (the "Who are the users?" slide).
USER_TYPES = ["invited", "dropped_off", "solo", "paid_teams", "together", "payment_errors", "planners"]
USER_NAMES = {f"cluster_{i}": name for i, name in enumerate(USER_TYPES)}
users = sessions.add_clusters(
    "user_type", features=[{"metric": "event_count_bulk", "metric_args": {"events": SESSIONS}}],
    method_args={"n_clusters": 7, "n_init": 30}, scaler="std", path_col="user_id",
).rename_segment_levels("user_type", USER_NAMES)


In [ ]:
#| echo: false
from toy import visits_as_events_figure

# A user whose first session repeats events (a vector of more than ones) and
# whose short session is a typical one: a look at the notifications.
HTML(visits_as_events_figure(taskboard, sessions, "user_00096"))


- A user's path is one long sequence of events, and it naturally splits into sessions
- **Idea:** treat each session as one event. To name it, look at what happens in it
- Vectorize each session as a **bag of events**: how many times each event happens in it
- Cluster the vectors with **K-Means**: each session gets one label
- The labels become the events: a path of dozens of events becomes a path of a few sessions


## What kind of session was it?

In [ ]:
#| echo: false
clusters = taskboard.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": 8, "n_init": 30},
    scaler="std",
    path_col="session_id",
    overview_metrics=[{"metric": "length"}, {"metric": "event_count_bulk"}],
)
clusters.cluster_renames = json.dumps(SESSION_NAMES)
clusters.slide_height = 600  # the whole slide below the title
clusters


- `teamwork`: comments, mentions, assignments: working it out with the team
- `focus`: long working sessions: tasks created, edited, completed
- `onboarding`: signup, the tour, a first project
- `checkout`: everyday work, then a checkout that goes through
- `checkout_failed`: a payment error and a support ticket
- `review`: reports, exports, automations, integrations
- `short`: a quick look: notifications, a task or two
- `planning`: tasks with due dates and dependencies, a look at the timeline


## Eight session types instead of 29 events

In [ ]:
#| echo: false
stage_graph(sessions, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

- After `onboarding`: → `focus` 38%, → `short` 34%; 15% never come back
- `path_start` → `short` 13%, → `teamwork` 10%: invited teammates skip onboarding
- `teamwork` → `teamwork` 22%: a team keeps working it out
- `planning` → `planning` 30%: some plan session after session
- `checkout_failed` → `checkout` 21%: a failed payment, then another try
- Before: the same paths, event by event


In [ ]:
#| echo: false
stage_graph(taskboard, view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

## The same paths, session by session

In [ ]:
#| echo: false
HTML(
    visit_paths_figure(sessions, USERS)
    + branching_figure(sessions.sample_paths(n=1000, random_state=7), 5, colors=taskboard_colors(sessions.df["event"]), size=250)
)

## Who are the users?

In [ ]:
#| echo: false
user_clusters = sessions.cluster_analysis(
    features=[{"metric": "event_count_bulk", "metric_args": {"events": SESSIONS}}],
    method_args={"n_clusters": 7, "n_init": 30},
    scaler="std",
    path_col="user_id",
    overview_metrics=[{"metric": "length"}, {"metric": "event_count_bulk"}],
)
user_clusters.cluster_renames = json.dumps(USER_NAMES)
user_clusters.slide_height = 600
user_clusters


- `invited`: came by invite: `short` and `teamwork` sessions, for a while
- `dropped_off`: `onboarding` and a session or two, then gone
- `solo`: work alone, in `focus` and `short` sessions
- `paid_teams`: every kind of session, a `checkout` among them
- `together`: many `teamwork` sessions: pairs, and the teammates they work with
- `payment_errors`: hit the broken payment form (`checkout_failed`); some paid on a later try
- `planners`: `planning` session after session


## What it looks like in Retentioneering code

In [ ]:
#| eval: false
#| code-line-numbers: "|5-10|11-12|13|15-21"
SESSIONS = ["teamwork", "focus", "onboarding", "checkout",
            "checkout_failed", "review", "short", "planning"]
sessions = (
    taskboard
    .add_clusters(
        "session_type",
        features=[{"metric": "event_count_bulk"}],
        method_args={"n_clusters": 8, "n_init": 30},
        scaler="std", path_col="session_id",
    )
    .collapse_events(group_col="session_id",
                     name={"col": "session_type"})
    .rename_events({f"cluster_{i}": v for i, v in enumerate(SESSIONS)})
)
users = sessions.add_clusters(
    "user_type",
    features=[{"metric": "event_count_bulk",
               "metric_args": {"events": SESSIONS}}],
    method_args={"n_clusters": 7, "n_init": 30},
    scaler="std", path_col="user_id",
)


- `add_clusters`: label every session (`path_col="session_id"`) with its k-means cluster, the ones `cluster_analysis` showed. The features: the bag of events, how many times each of the 29 events happens in the session. `n_init=30` keeps the clusters the same from run to run
- `collapse_events`: replace each session's events with a single event, named after its cluster
- `rename_events`: give the clusters the names we read off their profiles. Now `sessions.transition_graph()` shows the journey, session by session
- `add_clusters` again, on the sessions, per user (`path_col="user_id"`): users by the kinds of sessions they make


## What makes Retentioneering different

- **Built for clickstreams** with many distinct events, where every path is unique
- **Data processors** made for sequences: a line each to clean, cut, group and label paths
- **Visualizations made for paths**: transition graph, step matrix, cluster analysis, to find what a funnel can't see
- **Together they do more:** processors shape the paths, visualizations read them, as with the session types here

# Part 2: Retentioneering tools

## Sequence log data

Event log table

| user_id | session_id | event | timestamp | country | platform | n_session |
|---|---|---|---|---|---|---|
| user_0042 | sess_0042_1 | home | 2024-03-13 10:21:14 | US | desktop | 1 |
| user_0042 | sess_0042_1 | catalog | 2024-03-13 10:21:28 | US | desktop | 1 |
| user_0042 | sess_0042_1 | product_view | 2024-03-13 10:22:12 | US | desktop | 1 |
| user_0042 | sess_0042_2 | cart | 2024-03-15 19:02:40 | US | mobile | 2 |

- path identifiers: user_id, *session_id*
- event as a string: event
- event time: timestamp
- segments: *country*, *platform*
- miscellaneous: *n_session*

## Eventstream

- Eventstream: the core class – a data container and the API entry point
- A wrap of pandas.DataFrame
- Schema assigns column roles

In [ ]:
import pandas as pd

df = pd.read_csv("events.csv")
df

In [ ]:
#| code-line-numbers: "|2|3|4|5|6"
stream = rete.Eventstream(df, schema={
    "path_cols": ["user_id", "session_id"],
    "event_col": "event",
    "timestamp_col": "timestamp",
    "segment_cols": ["country", "platform"],
    "custom_cols": ["n_session"],
})

## Data accessor: Eventstream.df

In [ ]:
stream.df

- `event_type`: `raw` for your events; synthetic ones like `path_start` get their own type
- `index`: the event's position in its path
- `subindex`: orders event types that share a timestamp

## 2.1 Data processors

## Types of data processors

Data processor is a method that changes Eventstream and returns its modified version

- Data processors
  - Adding events
    - `add_events`
    - `add_start_end_events`
  - Renaming, editing & merging events
    - `rename_events`
    - `edit_events`
    - `collapse_events`
    - `urls_to_events`
  - Filtering, truncating & sampling
    - `drop_events`
    - `filter_events`
    - `filter_paths` 
    - `truncate_paths`
    - `sample_paths`
  - Sessions & lifecycle
    - `split_sessions`
    - `to_daily_states`
  - Segments & clustering
    - `add_segment`
    - `drop_segment`
    - `add_clusters`
    - `rename_segment_levels`

## Ten toy paths

In [ ]:
#| echo: false
from IPython.display import HTML
from toy import (
    edge_weights_figure,
    grouped_paths_figure,
    metrics_figure,
    pairs_figure,
    paths_figure,
    toy_events,
)

stream = rete.Eventstream(toy_events(), schema={"segment_cols": ["group"]})
HTML(grouped_paths_figure(stream, "group"))

## collapse_events: one event for a run

In [ ]:
one_b = stream.collapse_events(loops=True)

In [ ]:
#| echo: false
HTML(paths_figure(stream, one_b))

- `loops=True`: a run of the same event becomes one: B, B → B
- `group_col`: a run of rows sharing a value becomes one event; that is how sessions became events in Part 1
- `event_groups`, `bounds`: a run drawn from a set of events, or everything between two events

## Path metrics: one number per path

In [ ]:
metrics = stream.get_metrics([
    {"metric": "length"},
    {"metric": "has_event", "metric_args": {"event": "X"}},
    {"metric": "matches_pattern", "metric_args": {"pattern": "A->.*->B->.*->C"}},
])

In [ ]:
#| echo: false
HTML(metrics_figure(stream, metrics))

## Path patterns: regex over events

### Syntax

- `->`: then
- `.`: any one event
- `.*`: any run of events, including none
- `[A|D]`: one event, A or D
- `[^X]`: one event, anything but X
- `[^X]*`: a run of events with no X
- `path_start`, `path_end`: the path's own boundaries

### Examples

- `A->B`: A, then B right after it
- `A->.*->C`: A, then C at some point later
- `[A|D]->B`: A or D, then B
- `A->[^X]*->C`: A, then C, with no X in between
- `path_start->[^A|D]`: paths that begin with neither A nor D
- `C->path_end`: paths that end with C

Patterns aren't tied to a path's start or end: `A->.*->C` also matches B→A→C. To pin the start, write `path_start->A->.*->C`.

## truncate_paths: cut a window

In [ ]:
b_to_c = stream.truncate_paths(start_anchor="B", end_anchor="C")

In [ ]:
#| echo: false
HTML(paths_figure(stream, b_to_c, strike=True))

## Anchors: which B, and how far

In [ ]:
last_b = {"pattern": "B", "occurrence": "last"}
after_last_b = stream.truncate_paths(
    start_anchor=last_b,
    end_anchor={**last_b, "offset": 1},
)

In [ ]:
#| echo: false
HTML(paths_figure(stream, after_last_b, strike=True))

## Anchors: a list means whichever comes first

In [ ]:
b_to_c_or_end = stream.truncate_paths(start_anchor="B", end_anchor=["C", "path_end"])

In [ ]:
#| echo: false
HTML(paths_figure(stream, b_to_c_or_end, strike=True))

## add_events: mark a position

In [ ]:
marked = stream.add_events("BC", anchor={"pattern": "B->C", "at": "end"})

In [ ]:
#| echo: false
HTML(paths_figure(marked))

- A synthetic event `BC` where B is followed by C: now a step matrix can center on it, and a funnel can count it
- Other ways to add events: copy existing ones (`source_event`), any SQL (`sql`), or mark churn after days of inactivity (`churn`)

## Chaining data processors

In [ ]:
#| code-line-numbers: "|3|4|5"
clean = (
    stream
        .rename_events({"E": "C"})
        .drop_events(["X"])
        .filter_paths({"metric": "length", "op": ">=", "value": 3})
)

In [ ]:
#| echo: false
from toy import chain_steps_figure

HTML(chain_steps_figure(stream, clean))

## 2.2 Widgets: Funnel vs. Transition Graph

## Funnels: the traditional view

In [ ]:
#| output: false
stream.funnel(steps=["A", "B", "C"])

In [ ]:
#| echo: false
from toy import funnel_figure

HTML(funnel_figure(stream, ["A", "B", "C"]))

What a funnel can't see:

- Other routes: the 30% outside the funnel are a blank; their D→B→E is invisible
- Cycles: A→B→A→B→C counts the same as A→B→C
- Loops: A→B→B→C repeats B; the B→B step is invisible
- Detours: the X in A→X→B→C is invisible
- Other orders: B→A→C stops at A; the B before it is ignored

## Transition graph: count adjacent pairs

Every pair of neighboring events in every path, counted:

In [ ]:
#| echo: false
HTML(pairs_figure(stream))

In [ ]:
#| echo: false
stream.transition_graph(
    edge_weight="count",
    label_scale=2.5,
    sidebar_open=False,
    view={"edgeFilter": {"mode": "range", "range": [0, 1]}},
)

## Edge weights

In [ ]:
#| echo: false
HTML(edge_weights_figure(
    stream,
    edges=[("A", "B"), ("B", "C"), ("B", "B"), ("B", "E")],
    weights=["count", "unique_paths", "proba_out", "proba_in"],
))

- `count`: how many times the transition happened
- `unique_paths`: how many paths have it; A→B→A→B→C counts once
- `proba_out`: its share of everything that follows the source
- `proba_in`: its share of everything that leads into the target
- Also: `share_of_total`, `avg_per_path`, `time_median`, `time_q95`

## Diff mode: two segments, one graph

In [ ]:
#| echo: false
stage_graph(sessions, diff=("channel", "invite", "organic"), view={"edgeFilter": {"mode": "range", "range": [0, 1]}})

- `path_start` → `short` +45 pp, → `teamwork` +35 pp, → `onboarding` −100 pp: invited users skip onboarding and go straight to work
- `transition_graph(diff=("channel", "invite", "organic"))`: each edge is colored by how much more often one segment takes it than the other


## Interactive features (live demo)

- Views
- Search event
- Edge filter
- Node focus + 2-3 clicks
- Selecting traces
- Ego view
- user/session path selector

In [ ]:
#| output: false
ecom = rete.datasets.load_ecom()
checkout = ["cart", "shipping_details", "payment_details", "purchase"]
ecom.transition_graph(views=[
    {"name": "Checkout", "focus": {"type": "path", "nodes": checkout}},
    {"name": "Payment errors",
     "focus": {"type": "node", "event": "payment_error", "direction": "both"}},
])

## 2.3 Widgets: Tree vs. Step matrix 

## Step matrix: count events at each step

In [ ]:
#| echo: false
from toy import prefix_tree_svg, step_layout, step_matrix_table, step_paths_table

HTML(step_layout(
    step_paths_table(stream, 6, spotlight=(1, 2, 3)),
    step_matrix_table(stream, 6, spotlight=(1, 2, 3)),
))

## Tree vs. step matrix

In [ ]:
#| echo: false
HTML(step_layout(
    prefix_tree_svg(stream, 6, spotlight=(1, 2, 3)),
    step_matrix_table(stream, 6, spotlight=(1, 2, 3)),
))

## Step matrix centered on an event

In [ ]:
#| echo: false
HTML(step_layout(
    step_paths_table(stream, 3, spotlight=(0, -1, 1), pattern="B"),
    step_matrix_table(stream, 3, spotlight=(0, -1, 1), pattern="B"),
))

## Why not tree-based diagrams?

In [ ]:
#| echo: false
HTML(branching_figure(
    sessions.sample_paths(n=1000, random_state=7), 5,
    colors=taskboard_colors(sessions.df["event"]),
))

> A tree branches on every distinct beginning. On real data, almost every path gets a branch of its own.

## Step matrix: one row per session type (live demo)

The same 7,089 users, session by session: one row per session type, one column per step.

In [ ]:
sessions.step_matrix()

## Step matrix: diff mode

In [ ]:
#| eval: false
sessions.step_matrix(diff=("channel", "invite", "organic"))

In [ ]:
#| echo: false
diff = sessions.step_matrix(diff=("channel", "invite", "organic"))
diff.slide_height = 330
diff

- Each cell: the share in one segment minus the share in the other, step by step
- Invited users are in `short` (45%), `teamwork` (35%) or `focus` (18%) at step 1; organic ones go through `onboarding`, then `focus` and `short`


## 2.4 Local MCP server

## Start a local server

In [ ]:
#| eval: false
rete.mcp.serve(sessions, context={
    "description": "Taskboard, a task tracker for teams. channel: how a user came",
    "events": {"planning": "a session spent planning: tasks, due dates, dependencies"},
}, port=8765)


```bash
claude mcp add --transport sse retentioneering http://localhost:8765/sse
```

- The server runs in your Jupyter kernel: the data stay on your machine
- Any MCP agent connects over SSE: Claude Code, Codex, Cursor
- The agent works through tools: `describe`, `add_transition_graph`, `add_step_matrix`, `get_conversion_rate`, `export_report`

## Ask, and get a report

> How do invited teammates use Taskboard, compared with users who signed up themselves? Write a report.

```text
describe()
add_transition_graph(label="Session flow")
add_transition_graph(label="Invited vs organic", diff=["channel", "invite", "organic"])
check_analysis(analysis="...")
export_report(title="Invited vs organic users", analysis="...")
```

- In the analysis, every number comes with a link to where it was read: `[Invited vs organic:path_start->teamwork]`


## Every number is a link

In [ ]:
#| echo: false
from retentioneering.mcp import ReportSession, tools

ANALYSIS = """
## Invited users skip onboarding

[Invited vs organic:path_start->onboarding] `-100 pp`: every organic user starts with
onboarding, no invited one does.

## They go straight to work

[Invited vs organic:path_start->short] `+45 pp` and
[Invited vs organic:path_start->teamwork] `+35 pp`: a look at the notifications, or
a session with the team, is where an invited user starts.

## So

Teamwork holds on: [Session flow:teamwork->teamwork] `22%` of team sessions are
followed by another one. An invite brings a user into the team's work, not into the tour.
"""

session = ReportSession(sessions)
_ = tools.add_transition_graph(session, "Session flow")
_ = tools.add_transition_graph(session, "Invited vs organic", diff=["channel", "invite", "organic"])
_ = session.export("Invited vs organic users", ANALYSIS, "mcp_report.html")
HTML('<iframe class="report-frame" src="mcp_report.html"></iframe>')


<small>Click a link: the report opens its tab and focuses that edge.</small>